In [72]:
import json #import the json library so we can read the JSON files

In [73]:
def load_character(filepath): # load a character's data from a JSON file
    with open(filepath, "r") as f: # open the file in read mode
        data = json.load(f) # load the JSON data from the file into a Python dictionary
    return data
def get_normal_attack_multiplier(character_data, hit_number, talent_level): # get the multiplier for a specific hit of the normal attack at a given talent level
    hits = character_data["talents"]["normal_attack"]["hits"] # get the list of hits for the normal attack
    for hit in hits: # iterate through each hit in the list
        if hit["hit_number"] == hit_number: # check if the hit number matches the one we're looking for
            level_lookup = hit["multiplier_by_level"]   # open the folder — get all 11 sticky notes
            level_key = str(talent_level)               # translate 9 (numeral) into "9" (word), so it matches the labels
            multiplier = level_lookup[level_key]        # find the sticky note labeled "9", read its number
            return multiplier                           # hand that number back
    return None

def calculate_base_damage(multiplier,scaling_stat_value): # calculate the base damage of a hit based on its multiplier and the character's scaling stat value
    return (multiplier / 100) * scaling_stat_value # return the base damage by multiplying the multiplier (as a percentage) by the scaling stat value
def apply_defense_multiplier(damage, enemy_def, character_level): # apply the defense multiplier to the base damage based on the enemy's defense and the character's level
    def_multiplier = (character_level + 100) / (enemy_def + character_level + 100) # calculate the defense multiplier using the formula
    return damage * def_multiplier # return the final damage after applying the defense multiplier
def calculate_hit_damage(character_data, hit_number, enemy_def, character_level, talent_level): # calculate the final damage of a specific hit of the normal attack, taking into account the character's data, the hit number, the enemy's defense, and the character's level
    multiplier = get_normal_attack_multiplier(character_data, hit_number, talent_level) # get the multiplier for the specified hit number and talent level
    scaling_stat_name = character_data["talents"]["normal_attack"]["scaling_stat"] # get the name of the scaling stat used for the normal attack
    scaling_stat_value = character_data["base_stats"][scaling_stat_name] # get the value of the scaling stat from the character's base stats
    base = calculate_base_damage(multiplier, scaling_stat_value) # calculate the base damage using the multiplier and scaling stat value
    final = apply_defense_multiplier(base, enemy_def, character_level) # apply the defense multiplier to the base damage to get the final damage
    return final # return the final damage value

In [74]:
def get_elemental_skill_multiplier(character_data, crystal_shrapnel_stacks, talent_level): 
        stacks = character_data["talents"]["elemental_skill"]["stacks"] 
        for stack in stacks:
            if stack["stacks_consumed"] == crystal_shrapnel_stacks:
                shardshots = stack["shardshots"]
                dmg_bonus = stack["dmg_bonus"]
                pct_of_base = stack["pct_of_base"]
                return shardshots, dmg_bonus, pct_of_base
def get_elemental_skill_base_multiplier(character_data, talent_level):
    base_multiplier_by_level = character_data["talents"]["elemental_skill"]["base_multiplier_by_level"]
    multiplier_key = str(talent_level)
    base_multiplier = base_multiplier_by_level[multiplier_key]
    return base_multiplier
def calculate_skill_damage(character_data, crystal_shrapnel_stacks, talent_level, enemy_def, character_level):
    base_multiplier = get_elemental_skill_base_multiplier(character_data, talent_level)
    shardshots, dmg_bonus, pct_of_base = get_elemental_skill_multiplier(character_data, crystal_shrapnel_stacks, talent_level)
    
    scaled_multiplier = base_multiplier * (pct_of_base / 100)
    final_multiplier = scaled_multiplier * (1 + dmg_bonus / 100)
    
    scaling_stat_name = character_data["talents"]["elemental_skill"]["scaling_stat"]
    scaling_stat_value = character_data["base_stats"][scaling_stat_name]
    
    base_damage = calculate_base_damage(final_multiplier, scaling_stat_value)
    final = apply_defense_multiplier(base_damage, enemy_def, character_level)
    return final

In [75]:
navia_data = load_character("naviav3.json")
result = calculate_hit_damage(navia_data, hit_number="1", talent_level=1, enemy_def=100, character_level=90) # calls by specific hit number and talent level

In [76]:
navia_data=load_character("naviav3.json")
hit_number = navia_data["talents"]["normal_attack"]["hits"] # get the list of hits for the normal attack
total = 0 # initialize a variable to keep track of the total damage
for hit in hit_number: # iterate through each hit in the list
        result = calculate_hit_damage(navia_data, hit["hit_number"], talent_level=10, enemy_def=100, character_level=90) # calls full combo by talent level
        total = total + result
print(total)

1895.25


In [ ]:
navia_data = load_character("naviav3.json")
result = get_elemental_skill_multiplier(navia_data, crystal_shrapnel_stacks=6, talent_level=10)
print(result) # prints shardshots, dmg_bonus

(11, 45, 200)


In [78]:
navia_data = load_character("naviav3.json")
base = get_elemental_skill_base_multiplier(navia_data, talent_level=10)
print(base)

710.6


In [84]:
navia_data = load_character("naviav3.json")
result = calculate_skill_damage(navia_data, crystal_shrapnel_stacks=1, talent_level=10, enemy_def=100, character_level=90)
print(result)

2172.5847644827586
